# Calendar Process Steps

##### This code creates a csv file with dates along with its dependecies like year, month,day,ect. It asks you enter the desired dates (start and end dates)

In [ ]:
%%time

#--Time code processing
import time
start_time = time.perf_counter()

#--Step 1: Libraries
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

#--Step 2: Inputs parameters (start date, end date and index start)
start_date = '2020/01/01'
end_date = '2025/12/31' 
calendar_index = 1

#--Step 3: Dates parameters range list
dates = pd.date_range(start=start_date, end=end_date)
idx = list(range(calendar_index, calendar_index + len(dates)))

#--Step 4: Add months in spanish (only if needed)
es_month_initials = {
    1: 'E', 2: 'F', 3: 'M', 4: 'A', 5: 'M', 6: 'J',
    7: 'J', 8: 'A', 9: 'S', 10: 'O', 11: 'N', 12: 'D'
}

#--Step 5: DataFrame Columns
years = dates.year
months = dates.month
months_names = dates.month_name().str[:3].str.strip()
days = dates.day
days_week = dates.dayofweek + 1
days_name = dates.day_name().str[:3].str.strip()
months_ends = dates + pd.offsets.MonthEnd(0)
quarters = dates.quarter
week_start = dates.to_period('W-SAT').start_time
week_end = dates.to_period('W-SAT').end_time.date
weeks =  np.where(dates.dayofweek + 1 > 5, 'Weekend','Weekday')
week_year = dates.isocalendar().week
month_year =  dates.month_name().str[:3].str.strip() + "-" + dates.year.astype('str').str[-2:].str.strip()
ini_month = dates.month.map(es_month_initials)
max_year_val = np.where(dates.year.min() == dates.year.max() , dates.year.min(), dates.year.max())
max_month_val = np.where(dates.month.min() == dates.month.max(), dates.month.min(), dates.month.max())
max_week_val = week_start.max()

#--Step 6: Calendar Dataframe and its columns
calendar = pd.DataFrame({
    'idx': idx,
    'dates': dates,
    'years': years,
    'month': months,
    'months_names': months_names,
    'days': days,
    'days_week': days_week,
    'days_name': days_name,
    'months_ends': months_ends,
    'quarters': quarters,
    'week_start': week_start,
    'week_end': week_end,
    'weeks': weeks,
    'week_year': week_year
}).reset_index(drop=True)
#--Fix any incorrect column type
calendar = calendar.astype({
    'week_start': 'datetime64[ns]',
    'week_end': 'datetime64[ns]',
    #'week_year': 'int8',
    #'week_offset': 'int8'
})

#--Optional: Getting the minimun and maximun years of dataframe to create name file (optional)
min_year = calendar['years'].min()
max_year = calendar['years'].max()

#--Step 7: File Name and Output Folder Path
filename = "full_calendar.csv"
output_folder = r"Folder_Path" #<< Always check the folder path
os.makedirs(output_folder, exist_ok=True)
file_path = os.path.join(output_folder, filename)

#--Step 8: Dataframe to CSV file
calendar.to_csv(file_path, index=False, encoding='utf-8-sig')
print(f"File saved successfully as: {filename} with {len(calendar):,.0f} rows and {len(calendar.columns):,.0f} columns go and check your folder.")

end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Code took {total_time:.2f} seconds to complete.")

# Process code to create customer file and its columns

In [ ]:
%%time

#--Time code processing
import time
start_time = time.perf_counter()

#--Step 1: Libraries
from faker import Faker
import pandas as pd
import numpy as np
import os
import warnings
from unidecode import unidecode as ud
warnings.filterwarnings('ignore')
fake = Faker(['en_US','es_ES'])

#--Random seed and country
rng = np.random.default_rng(seed=100)
country = ['us','can']

#--Step 2: This function help to remove accents from spanish names/last name like {ñ,â,é}
def remove_accents(text):
    return ud(text)

#--Step 3: Calendar Path and DataFrame Columns
calendar = pd.read_csv(
    r"Folder_Path", usecols=['idx']
    )

#--Step 4: Inputs for Customer, quantity of rows, start index number
start_id = 1
number_customers = rng.integers(50000,80000,endpoint=True)
customer_idx = list(range(start_id, start_id + number_customers))
buyer_firstname = [remove_accents(fake.first_name()) for _ in range(number_customers)]
buyer_lastname = [remove_accents(fake.last_name()) for _ in range(number_customers)]
buyer_fullname =  [f'{f} {l}' for f,l in zip(buyer_firstname,buyer_lastname)] #Customer emails
buyer_email = [f'{f.lower()}{l.lower()}@techstore.com' for f,l in zip(buyer_firstname,buyer_lastname)] #Customer emails

#--Step 5: List of Variables from Series Frames, Probabilities and Random Choices
    #--Calendar idx as list, probability and choice
created_list = list(calendar['idx'])
prob_created = np.random.dirichlet(np.ones(len(created_list)))
date_idx = np.random.choice(created_list, size=number_customers, p=prob_created)

    #--Country list, probability and choice
country #<<--Full List
prob_country = np.random.dirichlet(np.ones(len(country)))
country_select = np.random.choice(country, size=number_customers, p=prob_country)

#--Step 6: Customer DataFrame
customer_df = pd.DataFrame({
    'idx': customer_idx,
    'customer_fullname':buyer_fullname,
    'customer_email': buyer_email,
    'customer_country': country_select,
    'created_on': date_idx
})

#--Step 7: Duplicated rows found
duplicated_rows = customer_df[['customer_email','customer_country']].duplicated().sum()
print(f"Total number of duplicate dropped rows: {duplicated_rows}")

#--Step 8: Drop Duplicates if appears
customer_df = (customer_df.sort_values('created_on',ascending=True)
           .drop_duplicates(subset=['customer_email'],keep='first')
           .reset_index(drop=True)
          )

#--Step 9: Reset idx column since it has gasps in-between
customer_df['idx'] = customer_df.index+1

#--Step 10: Convert data type of all DataFrame to lower memory usage
customer_df = customer_df.astype({
    'customer_country': 'category'})

#--Step 11: Output file name
filename_customers = 'full_customers.csv'

#--Step 12: File output folder path
output_folder = r"Folder_Path"
os.makedirs(output_folder, exist_ok=True)
file_customers = os.path.join(output_folder, filename_customers)

#--Step 13: CSV creation
customer_df.to_csv(file_customers, index=False, encoding='utf-8-sig')
print(f"File '{filename_customers}' was saved and created successfully with {len(customer_df):,.0f} rows and {len(customer_df.columns):,.0f} columns go and check your folder.")

end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Code took {total_time:.2f} seconds to complete.")

# Process code to create items device and sub categories

In [ ]:
%%time

#--Time code processing
import time
start_time = time.perf_counter()

#--Step 1: Libraries
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

#--Random seed and option list
rng = np.random.default_rng(seed=100)
payment_type = ['Credit card','Cash']
buy_type = ['In-store','Online']
purchase_state = ['Completed','Pending','Returned']

#--Step 2: Get files path to be used, these files help you to get needed information to create the output file
    #--Calendar Path and DataFrame Columns
calendar = pd.read_csv(
    r"Folder_Path"
    , usecols=['idx'])

    #--Customer Path and DataFrame Columns
customer = pd.read_csv(
    r"Folder_Path"
    , usecols=['idx','created_on'])

    #--Device and Device Categories Path and DataFrame Columns
devices = pd.read_csv(
    r"Folder_Path"
    , usecols=['idx','idx_catg'])

device_catg = pd.read_csv(r"Folder_Path"
    , usecols = ['idx','min_price','max_price'])

#--Step 3: Merging Device and Device Categories
    #--Merge Device Table with Device Categories to get prices
devices = pd.merge(
        devices,
        device_catg,
        left_on='idx_catg',
        right_on='idx',
        how='inner')
    #--Create a random Price column using the min and max columns values
devices['price'] = np.random.uniform(
    devices['min_price'],
    devices['max_price'],
    size=len(devices)
).round(2)
    #--Columns to drop
cols_dev_drop = ['idx_catg', 'idx_y','min_price','max_price']
devices = devices.rename(columns={'idx_x':'dev_idx'})
devices = devices.drop(cols_dev_drop, axis=1)

#--Step 4: Inputs for Device File, quantity of rows, start index number
number_rows_devices =  100 
start_index_devices = 1
dev_idx = list(range(start_index_devices, start_index_devices + number_rows_devices))

#--Step 5: List of Variables from Series Frames, Probabilities and Random Choices
    #--Calendar idx as list, probability and choice
list_date = list(calendar['idx'])
prob_date = np.random.dirichlet(np.ones(len(list_date)))
date_dev = np.random.choice(list_date, size=number_rows_devices, p=prob_date)

    #--Customer idx as list, probability and choice
prob_cust_names = np.random.dirichlet(np.ones(len(customer)))
cust_names = np.random.choice(len(customer), size=number_rows_devices, p=prob_cust_names)

    #--Devices idx as list, probability and choice
devices #<<--Full dataframe
prob_dev = np.random.dirichlet(np.ones(len(devices)))
device_idx = np.random.choice(devices.index, size=number_rows_devices, p=prob_dev)
device_sampled = devices.loc[device_idx]

    #--Device payments list, probability and choice
payment_type #<<--Full List
prob_dev_pay = np.random.dirichlet(np.ones(len(payment_type)))
dev_pay = np.random.choice(payment_type, size=number_rows_devices, p=prob_dev_pay)

    #--Refund state list, probability and choice
purchase_state #<<--Full List
prob_state = np.random.dirichlet(np.ones(len(purchase_state)))
order_state = np.random.choice(purchase_state, size=number_rows_devices, p=prob_state)

    #--Buy-Type list, probability and choice
buy_type #<<--Full List
prob_buy = np.random.dirichlet(np.ones(len(buy_type)))
bougth_select = np.random.choice(buy_type, size=number_rows_devices, p=prob_buy)

#--Step 7: Keep idx and created_on from the SAME selected rows
devices_df = customer.iloc[cust_names].copy()

#--Step 8: Rename Columns
devices_df = devices_df.rename(columns={'idx': 'customer_idx'}).reset_index(drop=True)

#--Step 9: Add additional columns
devices_df = devices_df.assign(
    bought_on = date_dev,
    bought_as = bougth_select,
    device_idx = device_sampled['dev_idx'].to_numpy(),
    device_price = device_sampled['price'].to_numpy(),
    device_pay = dev_pay,
    order_status = order_state
)

#--Step 10: Add filter column (customer cannot purchase on a date that is lower that their account creation date)
devices_df = devices_df[devices_df['created_on'] >= devices_df['bought_on']]

#--Step 11: Drop unnecessary columns and sort dataframe
devices_df = devices_df.drop(columns='created_on').reset_index(drop=True)
devices_df = devices_df.sort_values(by='bought_on',ascending=True)

# --Step 12: Convert data type of all DataFrame to lower memory usage
devices_df = devices_df.astype(
    {'bought_as': 'category',
    'device_price': 'float',
    'device_pay': 'category',
    'order_status': 'category'
    })

#--Step 13: Output file name
filename_device = 'full_device.csv'

#--Step 14: File output folder path
output_folder = r"Folder_Path"
os.makedirs(output_folder, exist_ok=True)
file_device = os.path.join(output_folder, filename_device)

#--Step 15: CSV creation
devices_df.to_csv(file_device, index=False, encoding='utf-8-sig')
print(f"File  was saved and created successfully as: {filename_device} with {len(devices_df):,.0f} rows and with {len(devices_df.columns):,.0f}, go and check your folder.")

end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Code took {total_time:.2f} seconds to complete.")

# Process code to create items apps and sub categories

In [ ]:
%%time

#--Time code processing
import time
start_time = time.perf_counter()

#--Random seed and option list
rng = np.random.default_rng(seed=100)
app_used = ['Apple','Google']
app_rate = [1,3,5]
country = ['us','can']

#--Step 1: Libraries
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

#--Step 2: Get files path to be used, these files help you to get needed information to create the output file
    #--Calendar Path and DataFrame Columns
calendar = pd.read_csv(
    r"Folder_Path"
    , usecols=['idx'])

    #--App Categories and Subcategories Paths and DataFrame Columns
apps_catgs = pd.read_csv(r"Folder_Path"
    , usecols = ['idx','min_price','max_price'])

app_subcatg = pd.read_csv(r"Folder_Path"
    , usecols = ['idx','idx_catg'])

#--Step 3: Merging App Table with App Categories
apps = pd.merge(
        app_subcatg,
        apps_catgs,
        left_on='idx_catg',
        right_on='idx',
        how='inner')
    #--Create a random Price column using the min and max columns values
apps['price'] = np.random.uniform(
    apps['min_price'],
    apps['max_price'],
    size=len(apps)
).round(2)
    #--Columns to drop
cols_app_drop = ['idx_catg', 'idx_y','min_price','max_price']
apps = apps.rename(columns={'idx_x':'app_idx'})
apps = apps.drop(cols_app_drop, axis=1)

#--Step 4: Inputs for App File, quantity of rows, start index number
start_index_app = 1
number_rows_app = rng.integers(50000,80000,endpoint=True)
app_idx = list(range(start_index_app, start_index_app + number_rows_app))

#--Step 5: List of Variables from Series Frames, Probabilities and Random Choices
    #--Calendar idx as list, probability and choice
list_date = list(calendar['idx'])
prob_date = np.random.dirichlet(np.ones(len(list_date)))
date_app = np.random.choice(list_date, size=number_rows_app, p=prob_date)

    #--App idx as list, probability and choice
apps #<<--Full dataframe
prob_app = np.random.dirichlet(np.ones(len(apps)))
app_idx = np.random.choice(apps.index, size=number_rows_app, p=prob_app) #<<--Get min and max index from dataframe
app_sampled = apps.loc[app_idx]

    #--App payments list, probability and choice
app_used #<<--Full List
prob_app_used = np.random.dirichlet(np.ones(len(app_used)))
app_device = np.random.choice(app_used, size=number_rows_app, p=prob_app_used)

    #--App rate list, probability and choice
app_rate #<<--Full List
prob_app_rate = np.random.dirichlet(np.ones(len(app_rate)))
app_rate = np.random.choice(app_rate, size=number_rows_app, p=prob_app_rate)

    #--Country list, probability and choice
country #<<--Full List
prob_country = np.random.dirichlet(np.ones(len(country)))
country_select = np.random.choice(country, size=number_rows_app, p=prob_country)

#--Step 6: App DataFrame
apps_df = pd.DataFrame({
    'download_on': date_app,
    'country':country_select,
    'apps_idx': app_sampled['app_idx'],
    'app_price': app_sampled['price'].to_numpy(),
    'app_dev': app_device,
    'app_rate': app_rate
})
#--Reset Index DataFrame
apps_df = apps_df.reset_index(drop=True)

#--Convert data type of all DataFrame to lower memory usage
apps_df = apps_df.astype({
    'country':'category',
    'app_price':'float',
    'app_dev':'category',
    'app_rate': 'int8'})

#--Step 7: Output file name
filename_apps = 'full_apps.csv'

#--Step 8: File output folder path
output_folder = r"Folder_Path"
os.makedirs(output_folder, exist_ok=True)
file_apps = os.path.join(output_folder, filename_apps)

#--Step 9: CSV creation
apps_df.to_csv(file_apps, index=False, encoding='utf-8-sig')
print(f"File was saved and created successfully as: {filename_apps} with {len(apps_df):,.0f} rows and {len(apps_df.columns):,.0f}, go and check your folder.")

end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Code took {total_time:.2f} seconds to complete.")